# Dot Plot Analysis: Per-Mode Visualizations

Dot plot visualizations of drum and foot onset phase distributions, organized separately by individual dance modes (group, individual, audience) for comparative analysis.

**Features**: Individual drum plots (Dun, J1, J2) and merged drum plots by mode, foot onset plots (left/right) by mode, KDE overlays, phase-aligned to musical cycles (0-400 metric units), stacked visualizations with subdivision markers, batch processing, optional image combination.

**Data**: Virtual cycle CSVs (`data/virtual_cycles/`), drum onset CSVs (`data/drum_onsets/`), foot onset CSVs (`data/logs_v*/.../onset_info/`), dance mode time segments (`data/dance_modes_ts/`), selected piece list (`data/selected_piece_list.pkl`).

**Output**: Individual drum plots (PNG) by mode and drum type, merged drum plots by mode, dance plots by mode, optional combined plots. Saved to `output_dot_plots/` with subdirectories for drum_single, drum_merged, dance, and combined analyses, organized by mode.

In [2]:
import os
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# from utils_subdivision.gen_distribution_single_plots import analyze_phases
# from utils_subdivision.gen_distribution_subplot import analyze_single_type    # plot_combined_results
# from utils_subdivision.gen_distribution_merged_plot import plot_merged
from utils_dot_plot.drum_single import analyze_phases
# from utils_dot_plot.drum_merged import plot_merged_per_mode

from utils_subdivision.gen_distribution_subplot import analyze_single_type
from utils_dot_plot.kinematic_dot_plot import *
from utils_dot_plot.drum_merged import *


base_output_dir = "output_dot_plots"

# Generate separate drum dot plot by Group, Individual or Audience

In [ ]:
m_idx = 2
mode = ["group", "individual", "audience"]
dance_mode = mode[m_idx]

with open('data/selected_piece_list.pkl', 'rb') as f:
    piece_list = pickle.load(f)
    
    
onset_types = ["Dun", "J1", "J2"]
base_output_dir = "output_dot_plots"
use_window = True
    
for file_name in piece_list:
    print(file_name)
    
    cycles_csv_path = f"data/virtual_cycles/{file_name}_C.csv"
    onsets_csv_path = f"data/drum_onsets/{file_name}.csv"
    dmode_path = f"data/dance_modes_ts/{file_name}_{dance_mode}.pkl"
    
    left_onset_path = f"data/logs_v4_0.007_foot_jun3/{file_name}_T/onset_info/{file_name}_T_left_foot_onsets.csv"
    right_onset_path = f"data/logs_v4_0.007_foot_jun3/{file_name}_T/onset_info/{file_name}_T_right_foot_onsets.csv"
    
    left_onsets = pd.read_csv(left_onset_path)["time_sec"].values
    right_onsets = pd.read_csv(right_onset_path)["time_sec"].values
    
        
    if os.path.exists(dmode_path):
        with open(dmode_path, "rb") as f:
            dance_mode_time_segments = pickle.load(f)       # list of tuples (start_time, end_time)
    else:
        continue

    for onset_type in onset_types:
        save_dir = os.path.join(base_output_dir, "drum_single", dance_mode, onset_type)
        os.makedirs(save_dir, exist_ok=True)
        
        # Update save path to include mode information
        if not use_window:
            save_path = os.path.join(save_dir, f"{file_name}_{onset_type}_full_duration_subplot.png")
        else:
            save_path = os.path.join(save_dir, f"{file_name}_{onset_type}_{mode[m_idx]}.png")
        
        # Analyze phases and save plot with dance_mode_time_segments
        analyze_phases(
            cycles_csv_path, onsets_csv_path, onset_type,
            dance_mode_time_segments=dance_mode_time_segments,
            dance_mode = dance_mode,
            save_path=save_path, 
            figsize= (10, 3), 
            dpi= 300,
            use_window= use_window   # to use time segments
        )
        
        print(f"Saved plot for {onset_type} to {save_path}")

# Generate merged drum dot plot by mode

In [ ]:
# Main execution code
m_idx = 2
mode = ["group", "individual", "audience"]
dance_mode = mode[m_idx]

with open('data/selected_piece_list.pkl', 'rb') as f:
    piece_list = pickle.load(f)
    
for file_name in piece_list:
    print(file_name)
    cycles_csv_path = f"data/virtual_cycles/{file_name}_C.csv"
    onsets_csv_path = f"data/drum_onsets/{file_name}.csv"
    dmode_path = f"data/dance_modes_ts/{file_name}_{dance_mode}.pkl"
    
    
    if os.path.exists(dmode_path):
        with open(dmode_path, "rb") as f:
            dance_mode_time_segments = pickle.load(f)
    else:
        continue
    

    # Call the modified function
    fig, ax, drum_phases_kde = drum_plot_merged_stacked(        # combined_xx, combined_h
        file_name=file_name,
        dance_mode=dance_mode,
        cycles_csv_path=cycles_csv_path,
        onsets_csv_path=onsets_csv_path,
        dance_mode_time_segments=dance_mode_time_segments,
        figsize=(10, 3),
        dpi=300,
        use_window=True,
        legend_flag=False
    )
    # plt.show()
    
    save_dir = os.path.join(base_output_dir, "drum_merged", dance_mode)
    os.makedirs(save_dir, exist_ok=True)
    
    save_path = os.path.join(save_dir, f"{file_name}_{dance_mode}_merged.png")
    plt.savefig(save_path, bbox_inches='tight', dpi=300)  # Add bbox_inches='tight' to prevent label clipping
    plt.close()


# Generate Dance dot plot by Group, Individual or Audience

In [ ]:
# m_idx = 0
modes = ["group", "individual", "audience"]
# dance_mode = modes[m_idx]

kde_data_per_dancer = {
    d: {m: [] for m in ["group", "individual", "audience"]}
    for d in ["dancer_1", "dancer_2", "dancer_3", "dancer_4"]
}

kde_data_all_pieces = {
    m: [] for m in ["gr", "in", "au"]
}


dancer_map = {
    ("E1", "D1"): "dancer_1",
    ("E1", "D5"): "dancer_1",
    ("E1", "D2"): "dancer_2",
    ("E2", "D3"): "dancer_3",
    ("E2", "D4"): "dancer_3",
    ("E3", "D5"): "dancer_4",
    ("E3", "D6"): "dancer_4",
}


with open('data/selected_piece_list.pkl', 'rb') as f:
    piece_list = pickle.load(f)

for dance_mode in modes:
    print(dance_mode)
    for file_name in piece_list:
        print(file_name)
        
        ensemble_name = file_name.split("_")[1]
        day = file_name.split("_")[2]

        
        cycles_csv_path = f"data/virtual_cycles/{file_name}_C.csv"
        dmode_path = f"data/dance_modes_ts/{file_name}_{dance_mode}.pkl"
        
        if not os.path.exists(dmode_path):
            continue
        
        left_onset_path = f"data/logs_v4_0.007_foot_jun3/{file_name}_T/onset_info/{file_name}_T_left_foot_onsets.csv"
        right_onset_path = f"data/logs_v4_0.007_foot_jun3/{file_name}_T/onset_info/{file_name}_T_right_foot_onsets.csv"
        
        left_onsets = pd.read_csv(left_onset_path)["time_sec"].values
        right_onsets = pd.read_csv(right_onset_path)["time_sec"].values
        
        if os.path.exists(dmode_path):
            with open(dmode_path, "rb") as f:
                dance_mode_time_segments = pickle.load(f)
                
            fig, ax, kde_dict = plot_foot_onsets_stacked(
                file_name=file_name,
                dance_mode=dance_mode,
                cycles_csv_path=cycles_csv_path,
                left_onsets=left_onsets,
                right_onsets=right_onsets,
                dance_mode_time_segments=dance_mode_time_segments,
                figsize=(10, 3),
                dpi=300,
                use_window=True,
                legend_flag=False
            )
            
        dancer = dancer_map.get((ensemble_name, day))
        if dancer:
            kde_data_per_dancer[dancer][dance_mode].append(kde_dict)
        
        # break
            # save_dir = os.path.join(base_output_dir, "dance", dance_mode)
            # os.makedirs(save_dir, exist_ok=True)
            
            # save_path = os.path.join(save_dir, f"{file_name}_{dance_mode}_merged.png")
            # plt.savefig(save_path, bbox_inches='tight', dpi=300)  # Add bbox_inches='tight' to prevent label clipping
        plt.close()
    # break

In [ ]:
# save
with open('output_dot_plots/feet_kde_data_per_dancer.pkl', 'wb') as f:
    pickle.dump(kde_data_per_dancer, f)

### Combine PNG vertically

In [9]:
import os
from PIL import Image

plot_1 = "output_dot_plots/Rainer-20Dec/hand_clap_by_piece"
plot_2 = "output_dot_plots/Rainer-20Dec/feet_by_piece"
save_dir = "output_dot_plots/Rainer-20Dec"

os.makedirs(save_dir, exist_ok=True)

# Get the intersection of filenames in both directories
dance_files = set(os.listdir(plot_1))
drum_files = set(os.listdir(plot_2))
# common_files = dance_files & drum_files

for fname1, fname2 in zip(sorted(dance_files), sorted(drum_files)):
    dance_img_path = os.path.join(plot_1, fname1)
    drum_img_path = os.path.join(plot_2, fname2)
    save_path = os.path.join(save_dir, "by_piece_" + fname1)

    # Open images
    img1 = Image.open(dance_img_path)
    img2 = Image.open(drum_img_path)

    # Make sure widths match (optional: resize if needed)
    if img1.width != img2.width:
        # Resize img2 to match img1 width, keeping aspect ratio
        new_height = int(img2.height * img1.width / img2.width)
        img2 = img2.resize((img1.width, new_height), Image.Resampling.LANCZOS)

    # Create new image with combined height
    total_height = img1.height + img2.height
    combined_img = Image.new('RGB', (img1.width, total_height), (255, 255, 255))
    combined_img.paste(img1, (0, 0))
    combined_img.paste(img2, (0, img1.height))

    # Save
    combined_img.save(save_path)
    print(f"Saved: {save_path}")

Saved: output_dot_plots/Rainer-20Dec/by_piece_Manjanin_au.png
Saved: output_dot_plots/Rainer-20Dec/by_piece_Maraka_au.png
Saved: output_dot_plots/Rainer-20Dec/by_piece_Suku_au.png
Saved: output_dot_plots/Rainer-20Dec/by_piece_Wasulunka_au.png


In [8]:
dance_files = set(os.listdir(plot_1))
drum_files = set(os.listdir(plot_2))
common_files = dance_files & drum_files

for fname1, fname2 in zip(sorted(dance_files), sorted(drum_files)):
    print(fname1, fname2)

Manjanin_au.png Manjanin_audience_combined.png
Maraka_au.png Maraka_audience_combined.png
Suku_au.png Suku_audience_combined.png
Wasulunka_au.png Wasulunka_audience_combined.png


In [ ]:
drum_files